# 🎨 Tune-Text AI Studio Cloud Worker (Google Colab T4 GPU)
### Tự động tạo Hình ảnh (Text-to-Image 8K) & Video chuyển động (Image-to-Video MP4)
---
Sổ tay này biến **Google Colab miễn phí** thành máy chủ Cloud GPU riêng biệt cho Tune-Text.
- **Không tốn 1% CPU/RAM của máy tính bạn**
- **Không kiểm duyệt, không giới hạn số lượng ảnh/video**
- **Không cần mở Chrome phụ hay cào web bên ngoài**

👉 **Hướng dẫn 1 bước duy nhất:** Bấm menu **Thời gian chạy (Runtime)** ➔ **Chạy tất cả (Run all)** (hoặc nhấn `Ctrl + F9`).

In [ ]:
# BƯỚC 1: CÀI ĐẶT THƯ VIỆN AI & CLOUDFLARE TUNNEL
!nvidia-smi
!pip install -q diffusers transformers accelerate safetensors huggingface_hub fastapi uvicorn pydantic torchvision Pillow imageio imageio-ffmpeg
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print("\n✅ Đã cài đặt xong toàn bộ thư viện cần thiết!")

In [ ]:
# BƯỚC 2: KHỞI TẠO MÔ HÌNH VÀ CHẠY SERVER FASTAPI TRỰC TIẾP TRONG COLAB
import os, io, time, base64, tempfile, threading, subprocess, re, urllib.request, torch, gc, traceback
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional
from PIL import Image
import uvicorn

# Khởi tạo FastAPI App
app = FastAPI(title="Tune-Text AI Studio", version="1.0.0")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32
pipe_image = None
pipe_video = None

def get_gpu_name():
    if torch.cuda.is_available():
        return f"{torch.cuda.get_device_name(0)} ({round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 1)} GB VRAM)"
    return "CPU"

def unload_model(pipe):
    if pipe is not None:
        del pipe
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return None

def load_image_pipeline():
    global pipe_image, pipe_video
    if pipe_image is not None:
        return pipe_image
    if pipe_video is not None:
        print("🔄 Đang giải phóng mô hình Video để nạp mô hình Vẽ Ảnh...")
        pipe_video = unload_model(pipe_video)
    
    print("⏳ Đang nạp mô hình Text-to-Image (SDXL Lightning)...\n")
    from diffusers import StableDiffusionXLPipeline, UNet2DConditionModel, EulerDiscreteScheduler
    from huggingface_hub import hf_hub_download
    base = "stabilityai/stable-diffusion-xl-base-1.0"
    repo = "ByteDance/SDXL-Lightning"
    ckpt = "sdxl_lightning_4step_unet.safetensors"
    unet = UNet2DConditionModel.from_config(base, subfolder="unet").to(DEVICE, DTYPE)
    unet.load_state_dict(torch.load(hf_hub_download(repo, ckpt), map_location=DEVICE))
    pipe_image = StableDiffusionXLPipeline.from_pretrained(base, unet=unet, torch_dtype=DTYPE, variant="fp16").to(DEVICE)
    pipe_image.scheduler = EulerDiscreteScheduler.from_config(pipe_image.scheduler.config, timestep_spacing="trailing")
    if torch.cuda.is_available():
        pipe_image.enable_attention_slicing()
        pipe_image.enable_vae_slicing()
    print("✅ Đã nạp thành công mô hình Text-to-Image! Tốc độ: ~2-3 giây/ảnh.\n")
    return pipe_image

def load_video_pipeline():
    global pipe_image, pipe_video
    if pipe_video is not None:
        return pipe_video
    if pipe_image is not None:
        print("🔄 Đang giải phóng mô hình Ảnh để nhường 100% VRAM cho mô hình Video...")
        pipe_image = unload_model(pipe_image)
    
    print("⏳ Đang nạp mô hình Image-to-Video (SVD)...\n")
    from diffusers import StableVideoDiffusionPipeline
    pipe_video = StableVideoDiffusionPipeline.from_pretrained("stabilityai/stable-video-diffusion-img2vid-xt", torch_dtype=DTYPE, variant="fp16").to(DEVICE)
    if torch.cuda.is_available():
        pipe_video.enable_attention_slicing()
        pipe_video.enable_vae_slicing()
    print("✅ Đã nạp thành công mô hình Image-to-Video! Tốc độ: ~25 giây/video.\n")
    return pipe_video

class ImgReq(BaseModel):
    prompt: str
    negative_prompt: Optional[str] = "ugly, blurry, low quality, distorted"
    aspect_ratio: Optional[str] = "16:9"
    steps: Optional[int] = 4

class VidReq(BaseModel):
    image_base64: str
    prompt: Optional[str] = ""
    num_frames: Optional[int] = 14
    fps: Optional[int] = 7

@app.get("/")
@app.get("/health")
@app.get("/info")
def health():
    return {"status": "ready", "gpu": get_gpu_name(), "models": ["SDXL-Lightning (8K Image)", "SVD-Video (MP4)"]}

@app.post("/api/generate-image")
def gen_image(req: ImgReq):
    pipe = load_image_pipeline()
    w, h = 1024, 576
    if req.aspect_ratio == "9:16": w, h = 576, 1024
    elif req.aspect_ratio == "1:1": w, h = 1024, 1024
    start = time.time()
    try:
        with torch.inference_mode():
            img = pipe(prompt=req.prompt, negative_prompt=req.negative_prompt, width=w, height=h, num_inference_steps=req.steps or 4, guidance_scale=0.0).images[0]
        buf = io.BytesIO()
        img.save(buf, format="PNG")
        b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
        return {"success": True, "image_base64": f"data:image/png;base64,{b64}", "elapsed_seconds": round(time.time() - start, 2)}
    except Exception as e:
        import traceback; traceback.print_exc()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        raise HTTPException(status_code=500, detail=f"Lỗi sinh ảnh: {str(e)}")

@app.post("/api/generate-video")
def gen_video(req: VidReq):
    try:
        import imageio
    except ImportError:
        import subprocess
        subprocess.run(["pip", "install", "-q", "imageio", "imageio-ffmpeg"], check=True)
        import imageio
    pipe = load_video_pipeline()
    from diffusers.utils import export_to_video
    raw = req.image_base64.split(",", 1)[1] if "," in req.image_base64 else req.image_base64
    inp = Image.open(io.BytesIO(base64.b64decode(raw))).convert("RGB").resize((1024, 576))
    start = time.time()
    try:
        with torch.inference_mode():
            frames = pipe(inp, decode_chunk_size=2, motion_bucket_id=127, fps=req.fps or 7, num_frames=req.num_frames or 14, num_inference_steps=18).frames[0]
        with tempfile.NamedTemporaryFile(suffix=".mp4", delete=False) as tf: tmp_p = tf.name
        export_to_video(frames, tmp_p, fps=req.fps or 7)
        with open(tmp_p, "rb") as f: vid_b = f.read()
        try: os.remove(tmp_p)
        except: pass
        b64 = base64.b64encode(vid_b).decode("utf-8")
        return {"success": True, "video_base64": f"data:video/mp4;base64,{b64}", "elapsed_seconds": round(time.time() - start, 2)}
    except Exception as e:
        import traceback; traceback.print_exc()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        raise HTTPException(status_code=500, detail=f"Lỗi sinh video: {str(e)}")

# Khởi động FastAPI ngầm trong luồng Colab an toàn 100%
def start_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()
print("⏳ Đang kích hoạt máy chủ FastAPI ngầm...")

# Nạp sẵn trước mô hình Ảnh vào GPU (Mô hình video chỉ nạp khi có yêu cầu để tiết kiệm 100% RAM)
print("⏳ Đang nạp sẵn Model Vẽ Ảnh (SDXL Lightning) vào GPU Tesla T4 (khoảng 30-40s)...\n")
try:
    load_image_pipeline()
except Exception as e:
    print(f"⚠️ Nạp model ảnh: {e}\n")

# Chờ và kiểm tra chắc chắn máy chủ phản hồi 200 trước khi mở Tunnel
server_ok = False
for _ in range(20):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=2) as r:
            if r.status == 200:
                server_ok = True
                print("✅ FastAPI Server trên Colab đã hoạt động!")
                break
    except Exception:
        time.sleep(1)

if not server_ok:
    print("❌ Lỗi: Server FastAPI chưa kịp phản hồi. Vui lòng bấm chạy lại ô này.")
else:
    # Dọn dẹp tiến trình tunnel cũ
    !pkill -f cloudflared > /dev/null 2>&1
    time.sleep(1)
    
    # Mở file ghi log tunnel để tránh đầy bộ đệm pipe gây lỗi 530
    log_tunnel = open("tunnel.log", "w")
    tunnel_proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stdout=log_tunnel, stderr=subprocess.STDOUT)
    print("⏳ Đang tạo đường dẫn Cloudflare Tunnel...")
    public_url = None
    for _ in range(40):
        time.sleep(0.5)
        if os.path.exists("tunnel.log"):
            with open("tunnel.log", "r") as f:
                c = f.read()
                m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', c)
                if m:
                    public_url = m.group(0)
                    break

    if public_url:
        print("\n===================================================================")
        print("🎉 CHÚC MỪNG BẠN! CLOUD GPU WORKER ĐÃ SẴN SÀNG 100%!")
        print(f"🔗 Đường dẫn kết nối Tune-Text: {public_url}")
        print("===================================================================")
        print("👉 Hãy sao chép đường dẫn trên và dán vào Tune-Text!")
        print("\n🟢 Server đang duy trì hoạt động ngầm (Vui lòng KHÔNG đóng tab Colab này)...")
        # Giữ cho cell Colab luôn chạy để không bị Colab tự ngắt kết nối
        try:
            while True:
                time.sleep(10)
        except KeyboardInterrupt:
            print("Đã dừng server.")
    else:
        print("⚠️ Chưa lấy được URL tự động, vui lòng thử lại ô này.")